# 06 · Missing data is part of the picture

**35 minutes · Intermediate.** Validate inputs, map reporting coverage across time, and prevent silence from being encoded as zero.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=06_Data_Quality.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## Four checks before a briefing
Require unique district/date keys, positive denominators, known district identifiers, and nonnegative reporting counts with received ≤ expected. Missing counts when no report was received must remain missing.
This exercise assumes **five reports expected per district per day**. A complete system would retain facility-level reporting records and distinguish late, revised, duplicate and zero-case reports.

In [ ]:
from fieldguide import validate_reports
districts,reports,facilities=load_tables()
validate_reports(reports,districts)
display(reports.loc[reports.cases_reported.isna()])
print("Validated rows:",len(reports))

In [ ]:
quality=reports.assign(completeness=100*reports.reports_received/reports.reports_expected)
matrix=quality.pivot(index="district_id",columns="date",values="completeness")
fig,ax=plt.subplots(figsize=(12,4))
im=ax.imshow(matrix,vmin=0,vmax=100,cmap="cividis",aspect="auto")
ax.set_yticks(range(len(matrix)),districts.set_index("district_id").loc[matrix.index,"district"])
ax.set_xticks(range(0,28,3),[d.strftime("%d %b") for d in matrix.columns[::3]])
ax.set(title="SYNTHETIC · Reporting completeness by district and day",xlabel="Report date (2025)")
fig.colorbar(im,ax=ax,label="Reports received (%)")
plt.tight_layout(); plt.show()

## Show the failure, then explain it
A duplicated row would double-count part of the numerator. Catch the problem before aggregation. The original table is preserved for the rest of the lab.

In [ ]:
bad=pd.concat([reports,reports.iloc[[0]]],ignore_index=True)
try:
    validate_reports(bad,districts)
except ValueError as error:
    print("Correctly rejected:",error)
missing=pd.Series([np.nan,np.nan])
print("Default sum (misleading for all-missing):",missing.sum())
print("sum(min_count=1), preserving unknown:",missing.sum(min_count=1))

## Completeness is a review signal
Separate quality flags from response priorities. Poor reporting can coincide with disrupted infrastructure and greater need. Do not automatically lower a district's priority simply because its data are incomplete.

In [ ]:
s=summarize()
s["needs_reporting_review"]=s.completeness_pct<80
display(s[["district","completeness_pct","cases","needs_reporting_review"]].round(1))
print("80% is a teaching threshold; select and justify local rules with responsible data owners.")

## Practice and debrief

Change the classroom completeness trigger from 80% to 90%. Identify the newly flagged districts and the verification workload created. Describe how you would handle a revised report without counting both versions.

## Sources and further learning

[CDC field investigation methods](https://www.cdc.gov/field-epi-manual/php/chapters/field-investigation.html) · [pandas sum and min_count](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html).